
<style>
:root{
  --sfi-ink:#0b1220; --sfi-muted:#475569; --sfi-line:#dbe3ef;
  --sfi-panel:#f8fafc; --sfi-blue:#1d4ed8; --sfi-gold:#b88a2e;
}
.sfi-hero{
  padding:22px 26px; border-radius:22px;
  background:radial-gradient(circle at 12% 18%, rgba(184,138,46,.22), transparent 24%),
             linear-gradient(135deg,#08111f 0%, #0b1c3d 52%, #102f6d 100%);
  color:white; border:1px solid rgba(255,255,255,.12);
  box-shadow:0 18px 42px rgba(15,23,42,.18);
}
.sfi-kicker{font-size:12px;text-transform:uppercase;letter-spacing:.18em;opacity:.78;margin-bottom:10px;}
.sfi-title{font-size:31px;font-weight:820;line-height:1.08;margin:0;}
.sfi-sub{font-size:15px;line-height:1.55;opacity:.9;margin-top:12px;max-width:980px;}
.sfi-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px;margin:14px 0;}
.sfi-card{
  border:1px solid var(--sfi-line); background:linear-gradient(180deg,#ffffff 0%,#f8fbff 100%);
  border-radius:16px; padding:14px 16px; box-shadow:0 8px 20px rgba(15,23,42,.055);
}
.sfi-card-dark{
  border:1px solid rgba(255,255,255,.12); background:linear-gradient(180deg,#0f172a 0%,#111d35 100%);
  color:white; border-radius:16px; padding:16px 18px;
}
.sfi-label{font-size:11px;text-transform:uppercase;letter-spacing:.11em;color:#475569;font-weight:700;}
.sfi-card-dark .sfi-label{color:rgba(255,255,255,.64);}
.sfi-metric{font-size:28px;font-weight:820;color:#102f6d;margin-top:2px;}
.sfi-card-dark .sfi-metric{color:#f6d78f;}
.sfi-copy{font-size:13px;line-height:1.52;color:#334155;margin-top:6px;}
.sfi-card-dark .sfi-copy{color:rgba(255,255,255,.78);}
.sfi-callout{padding:13px 15px;border-left:4px solid #1d4ed8;background:#eff6ff;border-radius:12px;color:#1e293b;}
.sfi-thesis{
  padding:16px 18px; border-radius:16px; background:linear-gradient(90deg,#fff7ed 0%,#eff6ff 100%);
  border:1px solid #dbeafe; color:#1e293b;
}
.sfi-small{font-size:12px;color:#64748b;line-height:1.45;}
.sfi-pill{display:inline-block;padding:5px 9px;border-radius:999px;border:1px solid #dbeafe;background:#eff6ff;color:#1e3a8a;font-size:12px;margin:2px 4px 2px 0;}
</style>

<div class="sfi-hero">
  <div class="sfi-kicker">CIMPS2026 · MIHM · Reproducible Evidence Notebook</div>
  <h1 class="sfi-title">Lectura homeostática de señales heterogéneas</h1>
  <div class="sfi-sub">
    Notebook autoejecutable para mostrar, con datos y diseño, cómo dos señales pueden compartir un mismo estado homeostático basal
    y aun así conservar perfiles internos diferentes de fricción, interferencia, coherencia, energía residual y variabilidad inducida.
  </div>
</div>

<div class="sfi-grid">
  <div class="sfi-card">
    <div class="sfi-label">Función</div>
    <div class="sfi-metric">Mostrar</div>
    <div class="sfi-copy">No solo calcular. Hacer visible la diferencia entre vector observado y estado resumido Φ.</div>
  </div>
  <div class="sfi-card">
    <div class="sfi-label">Método</div>
    <div class="sfi-metric">Contrastar</div>
    <div class="sfi-copy">Comparar señales mediante Fs, Di, Cs, Dcog, Er y Vi antes de interpretar el estado homeostático.</div>
  </div>
  <div class="sfi-card">
    <div class="sfi-label">Evidencia</div>
    <div class="sfi-metric">Trazar</div>
    <div class="sfi-copy">Exportar CSV/JSON y manifest SHA-256 para sostener reproducibilidad mínima.</div>
  </div>
</div>


In [ ]:

# 0. Configuración de entorno, estilo y dependencias
import sys, json, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML, Markdown

try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    WIDGETS_OK = True
except Exception:
    WIDGETS_OK = False

OUT_DIR = Path("outputs_mihm_sfi_premium")
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.precision", 4)

SFI_COLORS = {
    "ink": "#0b1220",
    "muted": "#475569",
    "line": "#dbe3ef",
    "panel": "#f8fafc",
    "blue": "#1d4ed8",
    "blue_dark": "#102f6d",
    "gold": "#b88a2e",
    "gold_light": "#f6d78f",
    "green": "#166534",
    "red": "#991b1b",
    "slate": "#334155"
}

SIGNAL_COLORS = {
    "REM618": "#102f6d",
    "easter_egg": "#b88a2e"
}

px.defaults.template = "plotly_white"
px.defaults.color_discrete_map = SIGNAL_COLORS

widget_state = "activo" if WIDGETS_OK else "no instalado en este entorno; el notebook conserva salida estática"
display(HTML("""
<div class="sfi-callout">
<b>Estado del notebook.</b><br>
Python {python_version} · Plotly activo · ipywidgets: {widget_state}.
</div>
""".format(python_version=sys.version.split()[0], widget_state=widget_state)))



## 1. Pregunta narrativa

<div class="sfi-thesis">
<b>Pregunta.</b> Si dos señales producen la misma lectura basal <b>Φ = 0.5000</b>, ¿son equivalentes?<br><br>
<b>Respuesta operativa.</b> No. Φ resume una condición de compensación. La diferencia observable vive en el vector:
<span class="sfi-pill">Fs</span>
<span class="sfi-pill">Di</span>
<span class="sfi-pill">Cs</span>
<span class="sfi-pill">Dcog</span>
<span class="sfi-pill">Er</span>
<span class="sfi-pill">Vi</span>
</div>


In [ ]:

# 1. Datos semilla reproducibles
records = [
    {"signal":"REM618",     "duration_s":150.400, "Fs":0.2159, "Di":0.6333, "Cs":0.9995, "Dcog":0.0040, "Er":0.4760, "Vi":0.3512},
    {"signal":"easter_egg", "duration_s":219.585, "Fs":0.2008, "Di":0.5359, "Cs":0.9991, "Dcog":0.0066, "Er":0.3090, "Vi":0.5641},
]
variables = ["Fs", "Di", "Cs", "Dcog", "Er", "Vi"]
labels_map = {
    "Fs": "Fricción sistémica",
    "Di": "Densidad de interferencia",
    "Cs": "Coherencia de señal",
    "Dcog": "Dispersión cognitiva",
    "Er": "Energía residual",
    "Vi": "Variabilidad inducida",
}
df = pd.DataFrame(records)

def sfi_table(dataframe, caption=None):
    if caption:
        display(HTML("<h4 style='margin-bottom:6px;color:#0b1220'>{}</h4>".format(caption)))
    display(
        dataframe.style
        .hide(axis="index")
        .format({c:"{:.4f}" for c in variables} | {"duration_s":"{:.3f}"})
        .set_table_styles([
            {"selector":"th", "props":[("background-color","#0f172a"),("color","white"),("font-weight","700"),("text-align","center")]},
            {"selector":"td", "props":[("border","1px solid #e2e8f0"),("text-align","center")]},
            {"selector":"table", "props":[("border-collapse","collapse"),("width","100%")]}
        ])
    )

sfi_table(df, "Tabla 1. Datos semilla del caso de uso")


/home/oai/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-0x0ytxz0 because there was an issue with the default path (/home/oai/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


signal,duration_s,Fs,Di,Cs,Dcog,Er,Vi
REM618,150.400,0.2159,0.6333,0.9995,0.0040,0.4760,0.3512
easter_egg,219.585,0.2008,0.5359,0.9991,0.0066,0.3090,0.5641



## 2. Estado basal y regla de interpretación

El notebook separa tres capas:

<div class="sfi-grid">
  <div class="sfi-card">
    <div class="sfi-label">Capa 1</div>
    <div class="sfi-metric">Señal</div>
    <div class="sfi-copy">Objeto observado: REM618 y easter_egg como señales no institucionales y reproducibles.</div>
  </div>
  <div class="sfi-card">
    <div class="sfi-label">Capa 2</div>
    <div class="sfi-metric">Vector</div>
    <div class="sfi-copy">Medición interna: Fs, Di, Cs, Dcog, Er y Vi.</div>
  </div>
  <div class="sfi-card">
    <div class="sfi-label">Capa 3</div>
    <div class="sfi-metric">Φ</div>
    <div class="sfi-copy">Lectura resumida de compensación homeostática, no sustituto del vector completo.</div>
  </div>
</div>

**Regla piloto**

\[
\Phi = 0.5000 \quad \text{si la señal es trazable, } C_s \geq 0.99,\; F_s < 0.30,\; D_{cog} < 0.01
\]

La regla es deliberadamente conservadora: evita sobreajustar dos observaciones iniciales.


In [ ]:

# 2. Regla piloto de Φ y tarjetas ejecutivas

def estimate_phi(row, traceable=True):
    if traceable and row["Cs"] >= 0.99 and row["Fs"] < 0.30 and row["Dcog"] < 0.01:
        return 0.5000, "basal_compensado"
    if not traceable:
        return np.nan, "no_interpretable"
    if row["Cs"] < 0.90 or row["Fs"] >= 0.60:
        return 0.2500, "degradado"
    return 0.4000, "transicion"

phi_results = df.apply(lambda r: estimate_phi(r), axis=1)
df["Phi"] = [x[0] for x in phi_results]
df["homeostatic_state"] = [x[1] for x in phi_results]

cards = []
for _, row in df.iterrows():
    cards.append("""
    <div class="sfi-card-dark">
      <div class="sfi-label">Señal observada</div>
      <div style="font-size:24px;font-weight:800;margin-top:2px;">{signal}</div>
      <div class="sfi-metric">Φ = {phi:.4f}</div>
      <div class="sfi-copy">Estado: <b>{state}</b></div>
      <div class="sfi-small" style="color:rgba(255,255,255,.7);margin-top:8px;">
        Cs={cs:.4f} · Fs={fs:.4f} · Dcog={dcog:.4f}
      </div>
    </div>
    """.format(signal=row["signal"], phi=row["Phi"], state=row["homeostatic_state"], cs=row["Cs"], fs=row["Fs"], dcog=row["Dcog"]))
display(HTML('<div class="sfi-grid">' + "".join(cards) + "</div>"))

fig = go.Figure()
domains = [(0.00, 0.48), (0.52, 1.00)]
for i, (_, row) in enumerate(df.iterrows()):
    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=row["Phi"],
        title={"text": row["signal"], "font":{"size":16}},
        domain={"x": list(domains[i]), "y": [0, 1]},
        number={"font":{"size":34}},
        gauge={
            "axis": {"range": [0, 1], "tickwidth":1},
            "bar": {"color": SIGNAL_COLORS[row["signal"]], "thickness":0.28},
            "bgcolor": "white",
            "borderwidth": 1,
            "bordercolor": "#dbe3ef",
            "steps": [
                {"range":[0,0.33], "color":"#fee2e2"},
                {"range":[0.33,0.66], "color":"#dbeafe"},
                {"range":[0.66,1.0], "color":"#dcfce7"},
            ],
            "threshold": {"line":{"color":"#0b1220","width":3}, "thickness":0.75, "value":0.5}
        }
    ))
fig.update_layout(
    title="Lectura resumida de estado homeostático",
    height=320,
    margin=dict(l=20,r=20,t=70,b=20),
    paper_bgcolor="white"
)
fig.show()



## 3. Lectura visual del vector

La visualización no debe adornar el análisis. Debe reducir carga cognitiva.

<div class="sfi-callout">
<b>Lectura esperada.</b> REM618 concentra más interferencia y energía residual. easter_egg concentra más variabilidad inducida. Ambas mantienen coherencia alta.
</div>


In [ ]:

# 3. Radar sobrio de perfiles vectoriales
radar_variables = variables
radar_labels = [labels_map[v] for v in radar_variables]

fig = go.Figure()
for _, row in df.iterrows():
    values = [row[v] for v in radar_variables]
    values += [values[0]]
    theta = radar_labels + [radar_labels[0]]
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=theta,
        fill="toself",
        name=row["signal"],
        opacity=0.58,
        line=dict(width=3, color=SIGNAL_COLORS[row["signal"]]),
        fillcolor=SIGNAL_COLORS[row["signal"]]
    ))

fig.update_layout(
    title="Radar MIHM: perfil interno de cada señal",
    polar=dict(
        bgcolor="#f8fafc",
        radialaxis=dict(visible=True, range=[0, 1], gridcolor="#dbe3ef"),
        angularaxis=dict(gridcolor="#e2e8f0")
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1),
    height=560,
    margin=dict(l=40,r=40,t=90,b=40)
)
fig.show()


In [ ]:

# 4. Heatmap para lectura ejecutiva
heat = df.set_index("signal")[variables].rename(columns=labels_map)
fig = px.imshow(
    heat,
    text_auto=".4f",
    aspect="auto",
    color_continuous_scale=[[0,"#f8fafc"],[0.5,"#bfdbfe"],[1,"#102f6d"]],
    title="Mapa de intensidad por variable"
)
fig.update_layout(height=330, margin=dict(l=20,r=20,t=60,b=20))
fig.update_xaxes(side="top")
fig.show()



## 4. Diferencia composicional

Aquí se vuelve explícita la diferencia entre señales.

- **REM618**: mayor \(D_i\) y \(E_r\).
- **easter_egg**: mayor \(V_i\).
- **Ambas**: \(C_s\) casi máximo.

Eso sostiene la tesis: mismo estado basal, distinta composición interna.


In [ ]:

# 5. Barras comparativas con anotación visual
long_df = df.melt(id_vars=["signal"], value_vars=variables, var_name="variable", value_name="value")
long_df["variable_label"] = long_df["variable"].map(labels_map)

fig = px.bar(
    long_df,
    x="variable_label", y="value", color="signal", barmode="group",
    text_auto=".4f",
    color_discrete_map=SIGNAL_COLORS,
    title="Comparación directa por variable"
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(
    height=500,
    xaxis_title="",
    yaxis_title="Valor normalizado",
    legend_title="Señal",
    margin=dict(l=20,r=20,t=70,b=90)
)
fig.show()



## 5. Widget de exploración controlada

El widget tiene una función precisa: permitir que un revisor o lector modifique el vector y observe si la regla piloto conserva el estado basal o cambia de régimen.

No es un simulador ornamental. Es una interfaz mínima para inspeccionar sensibilidad conceptual.


In [ ]:

# 6. Widget de alto impacto sobrio: exploración del vector
if WIDGETS_OK:
    signal_dd = widgets.Dropdown(options=df["signal"].tolist(), value="REM618", description="Señal")
    sliders = {
        v: widgets.FloatSlider(
            value=float(df.loc[df.signal=="REM618", v].iloc[0]),
            min=0, max=1, step=0.0001,
            description=v, readout_format=".4f",
            layout=widgets.Layout(width="95%")
        )
        for v in variables
    }
    out = widgets.Output()

    def current_payload():
        return {v: sliders[v].value for v in variables}

    def mini_radar(payload, signal_name):
        row = pd.Series({"signal": signal_name, **payload})
        phi, state = estimate_phi(row)
        vals = [payload[v] for v in variables] + [payload[variables[0]]]
        theta = [labels_map[v] for v in variables] + [labels_map[variables[0]]]
        fig = go.Figure()
        fig.add_trace(go.Scatterpolar(
            r=vals, theta=theta, fill="toself",
            line=dict(width=3, color=SIGNAL_COLORS.get(signal_name, "#102f6d")),
            fillcolor=SIGNAL_COLORS.get(signal_name, "#102f6d"),
            opacity=0.62,
            name=signal_name
        ))
        fig.update_layout(
            title="Vector interactivo · Φ={:.4f} · {}".format(phi, state),
            polar=dict(radialaxis=dict(visible=True, range=[0,1]), bgcolor="#f8fafc"),
            height=430,
            margin=dict(l=20,r=20,t=70,b=20)
        )
        return fig, phi, state

    def render_current(change=None):
        with out:
            clear_output(wait=True)
            payload = current_payload()
            fig, phi, state = mini_radar(payload, signal_dd.value)
            display(HTML("""
            <div class="sfi-card" style="margin-bottom:10px;">
              <div class="sfi-label">Resultado interactivo</div>
              <div class="sfi-metric">Φ = {:.4f}</div>
              <div class="sfi-copy">Estado: <b>{}</b></div>
            </div>
            """.format(phi, state)))
            fig.show()
            display(pd.DataFrame([{"signal": signal_dd.value, **payload, "Phi": phi, "state": state}])
                    .style.hide(axis="index").format({v:"{:.4f}" for v in variables} | {"Phi":"{:.4f}"}))

    def load_signal(change=None):
        row = df[df.signal == signal_dd.value].iloc[0]
        for v in variables:
            sliders[v].value = float(row[v])
        render_current()

    signal_dd.observe(load_signal, names="value")
    for sl in sliders.values():
        sl.observe(render_current, names="value")

    controls = widgets.VBox(
        [widgets.HTML("<b>Control vectorial MIHM</b>"), signal_dd] + [sliders[v] for v in variables],
        layout=widgets.Layout(width="42%", border="1px solid #dbe3ef", padding="12px")
    )
    panel = widgets.HBox([controls, out], layout=widgets.Layout(align_items="flex-start"))
    display(panel)
    render_current()
else:
    display(HTML("""
    <div class="sfi-callout" style="border-left-color:#b88a2e;background:#fff7ed;">
    <b>Widget preparado.</b><br>
    Este entorno no tiene <code>ipywidgets</code> instalado. En Colab/Jupyter se activa con:<br>
    <code>pip install ipywidgets</code><br><br>
    El notebook sigue siendo autoejecutable y conserva visualizaciones estáticas.
    </div>
    """))



## 6. Sensibilidad local

Monte Carlo no se usa aquí como validación absoluta. Se usa como prueba de estabilidad local del criterio piloto: pequeñas perturbaciones alrededor del vector base permiten observar si la lectura permanece en zona basal o cambia.


In [ ]:

# 7. Sensibilidad Monte Carlo
def monte_carlo(signal_name="REM618", seed=42, iterations=5000, sigma=0.025):
    rng = np.random.default_rng(seed)
    row = df[df["signal"] == signal_name].iloc[0]
    base = np.array([row[v] for v in variables], dtype=float)
    sims = np.clip(rng.normal(base, sigma, size=(iterations, len(variables))), 0, 1)
    states, phis = [], []
    for vec in sims:
        sim_row = pd.Series(dict(zip(variables, vec)))
        phi, state = estimate_phi(sim_row)
        states.append(state)
        phis.append(phi if not np.isnan(phi) else -1)
    return pd.DataFrame({"Phi": phis, "state": states})

mc = monte_carlo("REM618")
state_share = mc["state"].value_counts(normalize=True).rename_axis("state").reset_index(name="share")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=state_share["state"],
    y=state_share["share"],
    marker_color=["#102f6d" if s=="basal_compensado" else "#b88a2e" for s in state_share["state"]],
    text=[f"{v:.1%}" for v in state_share["share"]],
    textposition="outside"
))
fig.update_layout(
    title="Monte Carlo: proporción de estados bajo perturbación local",
    height=380,
    yaxis_title="Proporción",
    xaxis_title="Estado",
    margin=dict(l=20,r=20,t=70,b=50)
)
fig.show()

fig2 = px.histogram(mc, x="Phi", nbins=20, title="Distribución de Φ en simulación local")
fig2.update_traces(marker_color="#102f6d")
fig2.update_layout(height=360, margin=dict(l=20,r=20,t=60,b=40))
fig2.show()

display(state_share.style.hide(axis="index").format({"share":"{:.2%}"}))


state,share
transicion,62.16%
basal_compensado,37.84%



## 7. Trazabilidad verificable

El cierre del notebook no es una conclusión narrativa; es una salida verificable.  
Cada vector exportado genera hash SHA-256. Esto permite reconstruir si los datos usados para las figuras son los mismos que se reportan como evidencia.


In [ ]:

# 8. Exportación reproducible + SHA-256
def vector_hash(row):
    payload = {
        k: (float(row[k]) if k not in ["signal", "homeostatic_state"] else row[k])
        for k in ["signal","duration_s","Fs","Di","Cs","Dcog","Er","Vi","Phi","homeostatic_state"]
    }
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

df["sha256"] = df.apply(vector_hash, axis=1)

csv_path = OUT_DIR / "mihm_case_vectors_sfi_premium.csv"
json_path = OUT_DIR / "mihm_case_vectors_sfi_premium.json"
manifest_path = OUT_DIR / "manifest_sha256_sfi_premium.json"

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", force_ascii=False, indent=2)

manifest = {
    "model": "MIHM SFI premium storytelling notebook",
    "variables": variables + ["Phi"],
    "phi_rule": "Phi=0.5000 when traceable and Cs>=0.99 and Fs<0.30 and Dcog<0.01",
    "files": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in [csv_path, json_path]},
}
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

summary = pd.DataFrame([
    {"archivo": csv_path.name, "sha256": manifest["files"][csv_path.name]},
    {"archivo": json_path.name, "sha256": manifest["files"][json_path.name]},
])
display(summary.style.hide(axis="index").set_table_styles([
    {"selector":"th", "props":[("background-color","#0f172a"),("color","white"),("font-weight","700")]},
    {"selector":"td", "props":[("border","1px solid #e2e8f0")]}
]))

display(HTML("""
<div class="sfi-callout" style="border-left-color:#166534;background:#f0fdf4;">
<b>Exportación completada.</b><br>
{}<br>
{}<br>
{}
</div>
""".format(csv_path, json_path, manifest_path)))


archivo,sha256
mihm_case_vectors_sfi_premium.csv,e225942d4bee38eb73135882abff51e3c84efddb96a6fbf81740674291c7533b
mihm_case_vectors_sfi_premium.json,96c738d7a71679e2a2bb1820e16d67aa06c8888a4393a222e1da96016c84905f



## 8. Cierre ejecutivo

<div class="sfi-hero" style="box-shadow:none;">
  <div class="sfi-kicker">Lectura final</div>
  <div class="sfi-sub">
    El caso no muestra equivalencia entre señales. Muestra que dos señales pueden permanecer en una zona basal de compensación
    mientras expresan composiciones internas distintas. La evidencia principal no está solo en Φ; está en el vector, en su contraste,
    en su sensibilidad local y en la traza exportada.
  </div>
</div>
